<a href="https://colab.research.google.com/github/zaetae/regime-aware-ml-trading/blob/main/26_Conv1D_baseline_on_raw_price_windows.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 26. Conv1D baseline on raw price windows

**Question from the meeting:** does a small convolutional neural network, fed only the raw prices of the last 20 bars, do better than the Random Forest with hand-built indicators and patterns?

**Design**
- Input: the last 20 bars of Open, High, Low, Close, each expressed as a percentage of the most recent close (so the scale is the same for every stock), then standardised using training data only.
- Target: the same bar-by-bar triple-barrier label as before (long / short / no_trade), for every bar.
- Same forward split as notebooks 24 and 25: train before 2021-12-15, test from 2022-01-01. The last year of training is held out for early stopping (with a 16-day purge).
- One pooled network trained on all 24 tickers, 3 random seeds.
- Controls: (1) a Random Forest on the same raw windows, to separate "different input" from "different model", (2) the random baseline (stratified guessing), (3) the RF with indicators from notebook 25 (0.353 pooled, 0.366 per-instrument).
- Metric: F1 macro per ticker, averaged over tickers; also on event bars only.

**What to watch:** a small network on a noisy target can easily overfit, so we check the training and validation loss curves.

In [ ]:
import sys, os
for m in list(sys.modules.keys()):
    if m.startswith('src'):
        del sys.modules[m]

REPO_DIR  = '/content/regime-aware-ml-trading'
PROJ_ROOT = os.path.join(REPO_DIR, 'regime-aware-ml-trading')
if not os.path.isdir(PROJ_ROOT):
    os.system('git clone https://github.com/zaetae/regime-aware-ml-trading.git ' + REPO_DIR)
else:
    os.system(f'cd {REPO_DIR} && git pull -q')
os.system(f'{sys.executable} -m pip install -q yfinance hmmlearn scikit-learn seaborn statsmodels')

sys.path.insert(0, PROJ_ROOT)
os.chdir(PROJ_ROOT)

import pandas as pd, numpy as np, matplotlib.pyplot as plt
print("src exists:", os.path.isdir(os.path.join(PROJ_ROOT, 'src')))

In [ ]:
from numpy.lib.stride_tricks import sliding_window_view
from src.experiments.helpers import build_event_data, build_bar_data

ALL = ['SPY','QQQ','NVDA','GOOGL','AAPL','MSFT','AMZN','META','AVGO','TSLA','COST','ADBE',
       'AMD','NFLX','INTC','CSCO','JNJ','KO','WMT','PG','CAT','F','XOM','JPM']
W = 20

def to_dates(d):
    if 'dates' in d:
        return pd.to_datetime(d['dates'])
    return pd.to_datetime(d['features'].index)

Xs_list, meta_parts = [], []
for tk in ALL:
    try:
        e = build_event_data(tk)
        b = build_bar_data(tk, raw=e['raw'])
    except Exception as ex:
        print(tk, "FAILED:", ex); continue
    raw = e['raw']
    arr = raw[['Open', 'High', 'Low', 'Close']].values.astype(float)
    sw = sliding_window_view(arr, (W, 4))[:, 0]            # window ending at bar i+W-1
    rel = sw / sw[:, -1:, 3:4] - 1.0                        # relative to the last close of the window
    dates = to_dates(b)
    pos = raw.index.get_indexer(dates)
    ok = pos >= W - 1
    Xs_list.append(rel[pos[ok] - (W - 1)] * 100.0)          # in percent
    ev_dates = set(pd.to_datetime(e['labeled_df']['event_date']))
    meta_parts.append(pd.DataFrame({'ticker': tk, 'date': np.asarray(dates)[ok],
                                    'y': np.asarray(b['labels'])[ok],
                                    'is_event': pd.Series(np.asarray(dates)[ok]).isin(ev_dates).values}))
    print(tk, ok.sum(), "windows")

X = np.concatenate(Xs_list).astype('float32')
D = pd.concat(meta_parts, ignore_index=True)
print("\nX shape:", X.shape, "| label mix:", D.y.value_counts(normalize=True).round(3).to_dict())

In [ ]:
import tensorflow as tf
from tensorflow import keras
from sklearn.utils.class_weight import compute_class_weight

CUT = pd.Timestamp('2022-01-01'); PURGE = pd.Timedelta(days=16); VAL_START = pd.Timestamp('2020-12-15')
tr_all  = (D.date < CUT - PURGE).values
core    = (D.date < VAL_START - PURGE).values
val     = ((D.date >= VAL_START) & (D.date < CUT - PURGE)).values
test    = (D.date >= CUT).values
print("core / val / test sizes:", core.sum(), val.sum(), test.sum())

mu = X[core].mean(axis=(0, 1)); sd = X[core].std(axis=(0, 1))
Xn = (X - mu) / sd

classes = np.array(['long', 'no_trade', 'short'])
y_int = pd.Categorical(D.y, categories=classes).codes
cw = dict(enumerate(compute_class_weight('balanced', classes=np.arange(3), y=y_int[core])))

def build_cnn():
    m = keras.Sequential([
        keras.layers.Input(shape=(W, 4)),
        keras.layers.Conv1D(16, 3, padding='same', activation='relu'),
        keras.layers.Conv1D(16, 3, padding='same', activation='relu'),
        keras.layers.MaxPooling1D(2),
        keras.layers.Dropout(0.3),
        keras.layers.Flatten(),
        keras.layers.Dense(32, activation='relu'),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(3, activation='softmax'),
    ])
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy')
    return m

SEEDS = [42, 7, 123]
cnn_preds, histories = [], []
for seed in SEEDS:
    keras.utils.set_random_seed(seed)
    model = build_cnn()
    if seed == SEEDS[0]:
        print("parameters:", model.count_params())
    es = keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    h = model.fit(Xn[core], y_int[core], validation_data=(Xn[val], y_int[val]),
                  epochs=40, batch_size=256, class_weight=cw, callbacks=[es], verbose=0)
    histories.append(h.history)
    cnn_preds.append(classes[model.predict(Xn[test], batch_size=1024, verbose=0).argmax(1)])
    print(f"seed {seed}: stopped after {len(h.history['loss'])} epochs, "
          f"best val loss {min(h.history['val_loss']):.4f}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score

D_te = D[test].reset_index(drop=True)

def per_ticker_f1(pred, event_only=False, min_n=20):
    out = {}
    for tk, g in D_te.groupby('ticker'):
        m = g.index.values
        if event_only:
            m = m[g.is_event.values]
        if len(m) >= (15 if event_only else min_n):
            out[tk] = f1_score(D_te.y.values[m], pred[m], average='macro')
    return pd.Series(out)

# Random Forest on flattened raw windows (same input as the CNN)
rf_win = RandomForestClassifier(n_estimators=200, max_depth=8, class_weight='balanced',
                                random_state=42, n_jobs=-1).fit(X[tr_all].reshape(tr_all.sum(), -1), D.y[tr_all])
rf_pred = rf_win.predict(X[test].reshape(test.sum(), -1))

# stratified-guessing baseline, per ticker class frequencies from training
dummy_pred = np.empty(len(D_te), dtype=object)
for tk, g in D_te.groupby('ticker'):
    trn = D[tr_all & (D.ticker == tk).values]
    dm = DummyClassifier(strategy='stratified', random_state=42).fit(np.zeros((len(trn), 1)), trn.y)
    dummy_pred[g.index.values] = dm.predict(np.zeros((len(g), 1)))

def summarize(name, preds_list):
    allf = [per_ticker_f1(p) for p in preds_list]
    evf = [per_ticker_f1(p, event_only=True) for p in preds_list]
    means = [f.mean() for f in allf]; ev_means = [f.mean() for f in evf]
    return dict(model=name, f1_all_bars=np.mean(means), f1_std=np.std(means),
                f1_event_bars=np.mean(ev_means))

rows = [summarize('Conv1D (raw 20-bar OHLC), pooled', cnn_preds),
        summarize('Random Forest on raw 20-bar OHLC, pooled', [rf_pred]),
        summarize('Random baseline (stratified guessing)', [dummy_pred])]
res = pd.DataFrame(rows)
pd.options.display.float_format = '{:.3f}'.format
print(res.to_string(index=False))
print("\nReference from notebook 25 (RF with indicators, all bars): pooled 0.353, per-instrument 0.366, random 0.328")

# predicted class mix of the CNN vs the true mix
print("\nTrue label mix (test):", D_te.y.value_counts(normalize=True).round(3).to_dict())
print("CNN predicted mix:    ", pd.Series(cnn_preds[0]).value_counts(normalize=True).round(3).to_dict())

# overfitting check
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(histories[0]['loss'], label='train loss'); ax.plot(histories[0]['val_loss'], label='validation loss')
ax.set_xlabel('epoch'); ax.legend(); ax.set_title('Conv1D training curves (seed 42)')
plt.tight_layout(); plt.show()